# Tema 3.2: MaskAnnotator Avanzado y SAM2

*Duración estimada: 1 hora*

En esta clase tomaremos las máscaras que aprendimos a generar en la clase anterior y nos enfocaremos al 100% en cómo **visualizarlas, personalizarlas y manipularlas** usando `sv.MaskAnnotator`.

También introduciremos SAM2 para seguimiento temporal en video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
sam_path = "/content/drive/MyDrive/RandD/Archive_Zero_Resolved/sam3.pt"


In [ ]:
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)

urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg",    "assets/bus.jpg")
urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "assets/zidane.jpg")

In [ ]:
# Bloque de inicialización rápida (Recap de la clase anterior)
import supervision as sv
from ultralytics import YOLO, SAM
import cv2
import matplotlib.pyplot as plt

image = cv2.imread('assets/bus.jpg')
yolo_model = YOLO('yolov8n.pt')
sam_model = SAM(sam_path)

yolo_results = yolo_model(image)[0]
yolo_detections = sv.Detections.from_ultralytics(yolo_results)
sam_results = sam_model(image, bboxes=yolo_detections.xyxy.tolist())[0]
sam_detections = sv.Detections.from_ultralytics(sam_results)

print("Modelos y detecciones cargados exitosamente.")

## Paso 3: Visualizar con MaskAnnotator

In [ ]:
mask_annotator = sv.MaskAnnotator(opacity=0.6)
box_annotator = sv.BoxAnnotator()
# opacity controla cuánto "tapa" la máscara a la imagen original
# 0.6 → 60% máscara, 40% imagen original visible debajo
scene_yolo = box_annotator.annotate(scene=image.copy(), detections=yolo_detections)
annotated_sam = mask_annotator.annotate(scene=image.copy(), detections=sam_detections)
annotated_sam = box_annotator.annotate(scene=annotated_sam, detections=sam_detections)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.imshow(cv2.cvtColor(image.copy(), cv2.COLOR_BGR2RGB))
ax1.set_title("Solo bounding boxes (YOLO)")
ax1.axis("off")
ax2.imshow(cv2.cvtColor(annotated_sam, cv2.COLOR_BGR2RGB))
ax2.set_title("Máscaras de segmentación (SAM 3)")
ax2.axis("off")
plt.suptitle("Bounding box vs. segmentación a nivel de píxel", fontsize=13)
plt.tight_layout()
plt.show()


## Paso 3: Visualizar con MaskAnnotator

In [ ]:
mask_annotator = sv.MaskAnnotator(opacity=0.6)
# opacity controla cuánto "tapa" la máscara a la imagen original
# 0.6 → 60% máscara, 40% imagen original visible debajo

annotated_sam = mask_annotator.annotate(scene=image.copy(), detections=sam_detections)
annotated_sam = box_annotator.annotate(scene=annotated_sam, detections=sam_detections)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
ax1.imshow(cv2.cvtColor(scene_yolo, cv2.COLOR_BGR2RGB))
ax1.set_title("Solo bounding boxes (YOLO)")
ax1.axis("off")
ax2.imshow(cv2.cvtColor(annotated_sam, cv2.COLOR_BGR2RGB))
ax2.set_title("Máscaras de segmentación (SAM 3)")
ax2.axis("off")
plt.suptitle("Bounding box vs. segmentación a nivel de píxel", fontsize=13)
plt.tight_layout()
plt.show()


## 🔧 Exploración interactiva

### Experimento 1: Opacidad de las máscaras

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, opacity in zip(axes, [0.2, 0.5, 0.9]):
    ann = sv.MaskAnnotator(opacity=opacity)
    scene = ann.annotate(scene=image.copy(), detections=sam_detections)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(f"opacity={opacity}")
    ax.axis("off")
plt.tight_layout()
plt.show()
# 💭 Reflexión: ¿Cuándo conviene alta opacidad? ¿Cuándo baja?
# Alta opacidad → máscaras más visibles, objeto menos legible debajo.
# Baja opacidad → se ve el objeto original, pero la segmentación es menos clara.

### Experimento 2: Segmentar solo una clase

In [ ]:
# Filtrar antes de SAM para segmentar solo personas (clase 0)
# Combina filtrado (NB03) con segmentación
solo_personas = yolo_detections[yolo_detections.class_id == 0]
print(f"Personas detectadas: {len(solo_personas)}")

if len(solo_personas) > 0:
    bboxes_personas = solo_personas.xyxy.tolist()
    sam_personas_results = sam_model(image, bboxes=bboxes_personas)[0]
    det_personas = sv.Detections.from_ultralytics(sam_personas_results)
    
    scene = sv.MaskAnnotator(opacity=0.7).annotate(scene=image.copy(), detections=det_personas)
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("SAM solo en personas (filtrado antes de SAM)")
    plt.show()
# 💭 Reflexión: ¿Por qué filtramos antes de SAM en lugar de después?
# SAM es más lento que YOLO — procesar menos objetos ahorra tiempo de cómputo.

### Experimento 3: El mismo código en una imagen diferente

El pipeline de Supervision no cambia entre imágenes.
Este experimento demuestra el agnosticismo de framework:
el mismo código funciona para cualquier imagen de entrada.

In [ ]:
# Cargamos una imagen completamente diferente
image2 = cv2.imread("assets/zidane.jpg")

# El pipeline es IDÉNTICO al de bus.jpg — no cambia ni una línea
yolo_results2  = yolo_model(image2)[0]
yolo_det2      = sv.Detections.from_ultralytics(yolo_results2)
bboxes2        = yolo_det2.xyxy.tolist()
sam_results2   = sam_model(image2, bboxes=bboxes2)[0]
sam_det2       = sv.Detections.from_ultralytics(sam_results2)

annotated2 = sv.MaskAnnotator(opacity=0.6).annotate(scene=image2.copy(), detections=sam_det2)
annotated2 = box_annotator.annotate(scene=annotated2, detections=sam_det2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated2, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Mismo pipeline, imagen diferente — el código no cambió")
plt.show()
# 💭 Reflexión: ¿Necesitaste cambiar algo del pipeline para esta imagen?
# No — Supervision abstrajo la diferencia. Eso es exactamente lo que hace.

## El salto a SAM2: Segmentación temporal en Video
A diferencia de SAM (o SAM3 en imágenes estáticas), SAM2 fue diseñado con un motor de memoria temporal. Esto significa que puede retener la máscara de un objeto a lo largo de los frames de un video sin necesidad de recalcular todo.